# XWhy Text Explainer

The drop in fidelity for Example 2 occurs because **`predict` returns discrete binary step labels**, whereas **`predict_proba` (Examples 1 & 3) returns continuous probabilities**.

Here is why this fundamentally alters the surrogate fitting and metric scores:

* **Target Output Space ($y$)**
  * **Examples 1 & 3 (`predict_proba` / `__call__`)**: Output continuous confidence scores ($y \in [0.0, 1.0]$), such as $0.84$, $0.23$, or $0.91$.
  * **Example 2 (`predict`)**: Outputs hard discrete binary decisions ($y \in \{0, 1\}$), returning strictly $0$ or $1$.

* **Linear Surrogate Approximation**
  * The explainer fits a **linear regression surrogate model** (e.g., Ridge regression) over the perturbed samples: $\hat{y} = w_1 x_1 + w_2 x_2 + \dots + b$.
  * A linear equation fits continuous probability gradients smoothly and accurately, producing low error and a high $R^2$ ($0.9475$).
  * A linear equation cannot natively fit a hard step function ($0$ vs $1$). Forcing a linear hyperplane onto step-function target data creates large residual errors, lowering $R^2$ to $0.3121$ and increasing MSE.

* **Perturbation Step Jumps**
  * Removing a single word from text usually produces a small shift in continuous output (e.g., probability dropping from $0.52$ to $0.48$).
  * In discrete `predict` space, that exact same small shift crosses the decision boundary and causes a drastic output jump from $1$ to $0$. Linear models struggle to account for these step jumps without non-linear terms.

Using `predict_proba` (Examples 1 & 3) is always recommended when available. Preserving the confidence gradient gives local linear explainers like LIME and SHAP the continuous variance needed to calculate reliable, high-fidelity feature attributions.

## Setup

Run the next cell first on Google Colab or Kaggle (internet access required). It installs XWhy from GitHub; you can skip it if XWhy is already installed locally.

In [ ]:
# Setup for Google Colab / Kaggle: install the latest XWhy from GitHub.
# Skip this cell if XWhy is already installed in your environment.
%pip install -q "xwhy @ git+https://github.com/Dependable-Intelligent-Systems-Lab/xwhy.git"

from importlib.metadata import version
print("xwhy", version("xwhy"))

In [ ]:
import xwhy
from typing import List, Tuple
from sklearn.datasets import fetch_20newsgroups
import sklearn.feature_extraction.text as text
import sklearn.ensemble as ensemble
from sklearn.pipeline import make_pipeline

from xwhy.surrogate.types import SurrogateType
from xwhy.explainers.text import TextExplainer

## 1. Data Loading and Model Training

In [ ]:
def load_newsgroups_data(categories: List[str]) -> Tuple:
    """Load train and test subsets of 20 Newsgroups dataset."""
    train_data = fetch_20newsgroups(subset="train", categories=categories)
    test_data = fetch_20newsgroups(subset="test", categories=categories)
    return train_data, test_data

def build_vectorizer():
    """Create TF-IDF vectorizer."""
    return text.TfidfVectorizer(lowercase=False)

def train_random_forest(train_vectors, targets):
    """Train RandomForest classifier."""
    model = ensemble.RandomForestClassifier(n_estimators=500)
    model.fit(train_vectors, targets)
    return model

def build_pipeline(vectorizer, model):
    """Create sklearn pipeline."""
    return make_pipeline(vectorizer, model)

# Execute Setup
print("Loading data and training model...")
categories = ["alt.atheism", "soc.religion.christian"]
newsgroups_train, newsgroups_test = load_newsgroups_data(categories)

vectorizer = build_vectorizer()
train_vectors = vectorizer.fit_transform(newsgroups_train.data)

rf = train_random_forest(train_vectors, newsgroups_train.target)
pipeline_model = build_pipeline(vectorizer, rf)
print("Model training complete.\n")

## 2. Benchmark Configuration

In [ ]:
idx = 50
target_class_idx = 1  # "christian"
sample_count = 1000
document_text = newsgroups_test.data[idx]

## 3. Examples

### Example 1: Testing `predict_proba`

In [ ]:
print("--- Example 1: Testing predict_proba ---")
explainer_proba = TextExplainer(
    num_perturbations=sample_count,
    surrogate_type=SurrogateType.LIME,
    use_best_surrogate=False,
)

result_proba = explainer_proba.explain(
    instance=document_text,
    predict_fn=pipeline_model.predict_proba,
    class_index=target_class_idx,
)
print("Top 5 Features (predict_proba):", result_proba.coefficients[:5])
print("Metrics (predict_proba):", result_proba.metrics, "\n")

### Example 2: Testing `predict`

*Note: `predict()` returns a 1D array of class indices (0 or 1) rather than probabilities. `TextExplainer` will process this via the 1D array handling in your `explain()` method logic.*

In [ ]:
print("--- Example 2: Testing predict ---")
explainer_predict = TextExplainer(
    num_perturbations=sample_count,
    surrogate_type=SurrogateType.LIME,
    use_best_surrogate=False,
)

result_predict = explainer_predict.explain(
    instance=document_text,
    predict_fn=pipeline_model.predict,
    class_index=target_class_idx,
)
print("Top 5 Features (predict):", result_predict.coefficients[:5])
print("Metrics (predict):", result_predict.metrics, "\n")

### Example 3: Testing `__call__`

Passing a callable model wrapper object directly.

In [ ]:
print("--- Example 3: Testing __call__ ---")

# Define a custom wrapper class to simulate a callable model object
class CallableModelWrapper:
    def __init__(self, model):
        self.model = model
        
    def __call__(self, texts):
        return self.model.predict_proba(texts)

callable_model = CallableModelWrapper(pipeline_model)

explainer_call = TextExplainer(
    num_perturbations=sample_count,
    surrogate_type=SurrogateType.LIME,
    use_best_surrogate=False,
)

# Pass the callable object directly to the 'model' parameter
result_call = explainer_call.explain(
    instance=document_text,
    model=callable_model,
    class_index=target_class_idx,
)
print("Top 5 Features (__call__):", result_call.coefficients[:5])
print("Metrics (__call__):", result_call.metrics, "\n")

### Example 4: Direct Model Pipeline

Passing the model directly instead of predict_fn.

In [ ]:
explainer_direct = TextExplainer(
    num_perturbations=sample_count,
    surrogate_type=SurrogateType.LIME,
    use_best_surrogate=False,
)

result_direct = explainer_direct.explain(
    instance=document_text,
    model=pipeline_model,
    class_index=target_class_idx,
)
print("Top 5 Features (predict_proba):", result_direct.coefficients[:5])
print("Metrics (predict_proba):", result_direct.metrics, "\n")

## Visualizations

We can visualize the resulting feature contributions for the `predict_proba` run.

In [ ]:
result = result_direct

In [ ]:
result.plot()


# Visualizations

In [ ]:
xwhy.plots.text_heatmap(result)

In [ ]:
xwhy.plots.bar(result)

In [ ]:
xwhy.plots.waterfall(result)

In [ ]:
xwhy.plots.text(result)

In [ ]:
xwhy.plots.text_heatmap(result)

In [ ]:
xwhy.plots.force(result)

In [ ]:
xwhy.plots.decision(result)

In [ ]:
xwhy.plots.plot_feature_bar_chart(result)

In [ ]:
xwhy.plots.plot_feature_box_plot(result)